# Sistema de Recomendación de Medicamentos
**Filtrado Colaborativo basado en Reseñas Farmacéuticas - Dataset DrugLib**
## Joan David Martínez Hernández - 160004716


- **a.** Dado una condición médica -> medicamento con mayor rating.
- **b.** Top-5 medicamentos mejor valorados para una condición.
- **c.** Similitud entre medicamentos basada en perfiles de ratings por condiciones.
- **d.** Dado un medicamento -> condiciones más comunes para las que es formulado.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Asegurar directorios
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Cargar datasets
train_df = pd.read_csv('../data/drugLibTrain_raw.tsv', sep='\t', on_bad_lines='skip')
test_df = pd.read_csv('../data/drugLibTest_raw.tsv', sep='\t', on_bad_lines='skip')

print("Dimensiones de Entrenamiento:", train_df.shape)
print("Dimensiones de Prueba:", test_df.shape)


Dimensiones de Entrenamiento: (3107, 9)
Dimensiones de Prueba: (1036, 9)


In [2]:
# Unir datasets para el análisis y modelado
df = pd.concat([train_df, test_df], ignore_index=True)

# Limpiar columnas
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
elif '' in df.columns:
    df = df.rename(columns={'': 'id_original'})

# Renombrar urlDrugName a drugName
df = df.rename(columns={'urlDrugName': 'drugName'})

# Eliminar nulos en columnas clave
df = df.dropna(subset=['drugName', 'condition', 'rating'])

# Normalizar textos
df['drugName'] = df['drugName'].str.lower().str.strip()
df['condition'] = df['condition'].str.lower().str.strip()

# Convertir rating a numérico
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df.dropna(subset=['rating'])

# Eliminar condiciones con valores numéricos sin sentido (ej: dígitos solos)
df = df[~df['condition'].str.contains(r'^\d+$', na=True)]

print("Dimensiones después de limpieza:", df.shape)
print(df.head())


Dimensiones después de limpieza: (4141, 8)
           drugName  rating         effectiveness          sideEffects  \
0         enalapril       4      Highly Effective    Mild Side Effects   
1  ortho-tri-cyclen       1      Highly Effective  Severe Side Effects   
2           ponstel      10      Highly Effective      No Side Effects   
3          prilosec       3  Marginally Effective    Mild Side Effects   
4            lyrica       2  Marginally Effective  Severe Side Effects   

                                condition  \
0  management of congestive heart failure   
1                        birth prevention   
2                        menstrual cramps   
3                             acid reflux   
4                            fibromyalgia   

                                      benefitsReview  \
0  slowed the progression of left ventricular dys...   
1  Although this type of birth control has more c...   
2  I was used to having cramps so badly that they...   
3  The acid reflu

In [3]:
# Configurar tema de graficación
sns.set_theme(style="whitegrid")
plt.rcParamás['figure.figsize'] = (10, 6)

# a) Distribución de ratings
plt.figure(figsize=(8, 4))
sns.histplot(df['rating'], bins=10, kde=True, color='skyblue')
plt.title('Distribución de Ratings')
plt.xlabel('Rating (1-10)')
plt.ylabel('Frecuencia')
plt.savefig('../outputs/eda_rating_dist.png', dpi=150, bbox_inches='tight')
plt.show()

# b) Top 15 condiciones médicas más comunes
top_conditions = df['condition'].value_counts().head(15)
plt.figure(figsize=(10, 5))
sns.barplot(x=top_conditions.values, y=top_conditions.index, palette='viridis', hue=top_conditions.index, legend=False)
plt.title('Top 15 Condiciones Médicas Más Frecuentes')
plt.xlabel('Cantidad de Reseñas')
plt.ylabel('Condición')
plt.savefig('../outputs/eda_top_conditions.png', dpi=150, bbox_inches='tight')
plt.show()

# c) Top 15 medicamentos más reseñados
top_drugs = df['drugName'].value_counts().head(15)
plt.figure(figsize=(10, 5))
sns.barplot(x=top_drugs.values, y=top_drugs.index, palette='magma', hue=top_drugs.index, legend=False)
plt.title('Top 15 Medicamentos Más Reseñados')
plt.xlabel('Cantidad de Reseñas')
plt.ylabel('Medicamento')
plt.savefig('../outputs/eda_top_drugs.png', dpi=150, bbox_inches='tight')
plt.show()

# d) Boxplot de rating por efectividad
effectiveness_order = ['Ineffective', 'Marginally Effective', 'Moderately Effective', 'Considerably Effective', 'Highly Effective']
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='effectiveness', y='rating', order=effectiveness_order, palette='Set2')
plt.title('Distribución de Rating según la Efectividad')
plt.xlabel('Efectividad')
plt.ylabel('Rating')
plt.savefig('../outputs/eda_rating_effectiveness.png', dpi=150, bbox_inches='tight')
plt.show()


C:\Users\ESTUDIANTE\AppData\Local\Temp\ipykernel_19688\3642455175.py:37: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='effectiveness', y='rating', order=effectiveness_order, palette='Set2')


In [4]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import pickle

# Matriz Condición-Medicamento
pivot = df.pivot_table(
    index='condition',
    columns='drugName',
    values='rating',
    aggfunc='mean'
)

print(f"Matriz Condición x Medicamento: {pivot.shape}")

# Transponer para similitud de medicamentos
pivot_drug = pivot.T.fillna(0)

# Normalizar
scaler = StandardScaler()
pivot_scaled = scaler.fit_transform(pivot_drug)

# Calcular similitud coseno entre medicamentos
drug_sim_matrix = cosine_similarity(pivot_scaled)
drug_sim_df = pd.DataFrame(
    drug_sim_matrix,
    index=pivot_drug.index,
    columns=pivot_drug.index
)

# Guardar la matriz
with open('../models/similarity_matrix.pkl', 'wb') as f:
    pickle.dump(drug_sim_df, f)

print("Matriz de similitud coseno creada y guardada.")


Matriz Condición x Medicamento: (1806, 541)
Matriz de similitud coseno creada y guardada.


In [5]:
def recomendar_mejor_medicamento(condition, df, min_reviews=5):
    condition = condition.lower().strip()
    subset = df[df['condition'] == condition]
    if subset.empty:
        return f"No se encontraron datos para la condición: '{condition}'"
    
    resumen = subset.groupby('drugName').agg(
        avg_rating=('rating', 'mean'),
        n_reviews=('rating', 'count')
    ).reset_index()
    
    # Filtrar por mínimo de reseñas
    resumen_filtered = resumen[resumen['n_reviews'] >= min_reviews]
    if resumen_filtered.empty:
        resumen_filtered = resumen
        
    best = resumen_filtered.loc[resumen_filtered['avg_rating'].idxmax()]
    return best

print("=== PRUEBA a.1: depression ===")
print(recomendar_mejor_medicamento('depression', df))

print("\n=== PRUEBA a.2: breast cancer ===")
print(recomendar_mejor_medicamento('breast cancer', df))

def plot_top_medicamentos_condicion(condition, df, top_n=10, min_reviews=3):
    condition = condition.lower().strip()
    subset = df[df['condition'] == condition]
    resumen = subset.groupby('drugName').agg(
        avg_rating=('rating', 'mean'),
        n_reviews=('rating', 'count')
    ).reset_index()
    resumen = resumen[resumen['n_reviews'] >= min_reviews].nlargest(top_n, 'avg_rating')
    
    plt.figure(figsize=(10, 5))
    bars = plt.barh(resumen['drugName'], resumen['avg_rating'], color='steelblue')
    plt.xlabel('Rating Promedio')
    plt.ylabel('Medicamento')
    plt.title(f'Top {top_n} Medicamentos para: {condition.title()}')
    plt.xlim(0, 11)
    for bar, n in zip(bars, resumen['n_reviews']):
        plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'n={n}', va='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(f'../outputs/top_drugs_{condition.replace(" ","_")}.png', dpi=150)
    plt.show()

plot_top_medicamentos_condicion('depression', df)
plot_top_medicamentos_condicion('breast cancer', df)


=== PRUEBA a.1: depression ===
drugName       pristiq
avg_rating    8.666667
n_reviews            6
Name: 18, dtype: object

=== PRUEBA a.2: breast cancer ===
drugName      aromasin
avg_rating         9.0
n_reviews            1
Name: 0, dtype: object


In [6]:
def top5_medicamentos(condition, df, min_reviews=3):
    condition = condition.lower().strip()
    subset = df[df['condition'] == condition]
    if subset.empty:
        return f"No se encontraron datos para: '{condition}'"
    
    resumen = subset.groupby('drugName').agg(
        avg_rating=('rating', 'mean'),
        n_reviews=('rating', 'count')
    ).reset_index()
    
    resumen = resumen[resumen['n_reviews'] >= min_reviews]
    top5 = resumen.nlargest(5, 'avg_rating').reset_index(drop=True)
    top5.index += 1
    return top5

print("=== PRUEBA b.1: allergies ===")
print(top5_medicamentos('allergies', df))

print("\n=== PRUEBA b.2: anxiety ===")
print(top5_medicamentos('anxiety', df))

def plot_top5_static(condition, df):
    top5 = top5_medicamentos(condition, df)
    if isinstance(top5, str):
        print(top5)
        return
    plt.figure(figsize=(8, 4))
    sns.barplot(x='avg_rating', y='drugName', data=top5, palette='Blues_r', hue='drugName', legend=False)
    plt.xlabel('Rating Promedio')
    plt.ylabel('Medicamento')
    plt.title(f'Top 5 Medicamentos — {condition.title()}')
    plt.xlim(0, 11)
    for i, row in enumerate(top5.itertuples()):
        plt.text(row.avg_rating + 0.1, i, f"{row.avg_rating:.2f} (n={row.n_reviews})", va='center')
    plt.tight_layout()
    plt.savefig(f'../outputs/top5_{condition.replace(" ","_")}.png', dpi=150)
    plt.show()

plot_top5_static('allergies', df)
plot_top5_static('anxiety', df)


=== PRUEBA b.1: allergies ===
   drugName  avg_rating  n_reviews
1   nasonex    8.666667          3
2   flonase    8.250000          8
3   allegra    8.111111          9
4    zyrtec    7.400000         10
5  claritin    5.166667          6

=== PRUEBA b.2: anxiety ===
     drugName  avg_rating  n_reviews
1  alprazolam    9.400000          5
2      zoloft    9.400000          5
3      buspar    8.666667          3
4       paxil    8.571429          7
5      valium    8.333333          6


In [7]:
def similitud_drogas(drug1, drug2, drug_sim_df):
    drug1 = drug1.lower().strip()
    drug2 = drug2.lower().strip()
    if drug1 not in drug_sim_df.index:
        return f"Medicamento '{drug1}' no encontrado."
    if drug2 not in drug_sim_df.index:
        return f"Medicamento '{drug2}' no encontrado."
    sim = drug_sim_df.loc[drug1, drug2]
    return f"Similitud coseno entre '{drug1}' y '{drug2}': {sim:.4f}"

def top_similares(drug, drug_sim_df, top_n=10):
    drug = drug.lower().strip()
    if drug not in drug_sim_df.index:
        return f"'{drug}' no encontrado."
    sim_series = drug_sim_df[drug].drop(index=drug).sort_values(ascending=False)
    return sim_series.head(top_n).reset_index()

print("=== PRUEBA c.1: lyrica vs gabapentin ===")
print(similitud_drogas('lyrica', 'gabapentin', drug_sim_df))

print("\n=== PRUEBA c.2: prozac vs zoloft ===")
print(similitud_drogas('prozac', 'zoloft', drug_sim_df))

print("\n=== Medicamentos más similares a Lyrica ===")
print(top_similares('lyrica', drug_sim_df))

def plot_heatmap_similitud(drugs_list, drug_sim_df):
    valid_drugs = [d.lower().strip() for d in drugs_list if d.lower().strip() in drug_sim_df.index]
    sub = drug_sim_df.loc[valid_drugs, valid_drugs]
    plt.figure(figsize=(8, 6))
    sns.heatmap(sub, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
    plt.title('Heatmap de Similitud entre Medicamentos')
    plt.tight_layout()
    plt.savefig('../outputs/heatmap_similitud.png', dpi=150)
    plt.show()

sample_drugs = ['lyrica', 'prozac', 'zoloft', 'lexapro', 'cymbalta', 'gabapentin', 'xanax']
plot_heatmap_similitud(sample_drugs, drug_sim_df)


=== PRUEBA c.1: lyrica vs gabapentin ===
Medicamento 'gabapentin' no encontrado.

=== PRUEBA c.2: prozac vs zoloft ===
Similitud coseno entre 'prozac' y 'zoloft': 0.0130

=== Medicamentos más similares a Lyrica ===
     drugName    lyrica
0       actiq  0.071513
1   neurontin  0.020464
2     ritalin  0.012225
3    ultracet  0.011653
4      lortab  0.011163
5     ecotrin  0.010282
6  darvocet-n  0.008492
7    percocet  0.007580
8     desyrel  0.007511
9     relafen  0.007304


In [8]:
def condiciones_por_medicamento(drug, df, top_n=5):
    drug = drug.lower().strip()
    subset = df[df['drugName'] == drug]
    if subset.empty:
        return f"No se encontraron datos para el medicamento: '{drug}'"
    resumen = subset.groupby('condition').agg(
        n_prescripciones=('condition', 'count'),
        avg_rating=('rating', 'mean')
    ).reset_index().nlargest(top_n, 'n_prescripciones')
    return resumen

print("=== PRUEBA d.1: lyrica ===")
print(condiciones_por_medicamento('lyrica', df))

print("\n=== PRUEBA d.2: prozac ===")
print(condiciones_por_medicamento('prozac', df))

def plot_condiciones_static(drug, df):
    result = condiciones_por_medicamento(drug, df)
    if isinstance(result, str):
        print(result)
        return
    plt.figure(figsize=(6, 6))
    plt.pie(result['n_prescripciones'], labels=result['condition'], autopct='%1.1f%%', colors=sns.color_palette('pastel'))
    plt.title(f'Condiciones para las que se prescribe: {drug.title()}')
    plt.tight_layout()
    plt.savefig(f'../outputs/condiciones_{drug}.png', dpi=150)
    plt.show()

plot_condiciones_static('lyrica', df)
plot_condiciones_static('prozac', df)


=== PRUEBA d.1: lyrica ===
                        condition  n_prescripciones  avg_rating
2                    fibromyalgia                12    5.166667
5                     fibromylgia                 3    6.666667
0                    chronic pain                 2    8.000000
11                pain in my legs                 2    5.000000
1   complex regional pain sydrome                 1    8.000000

=== PRUEBA d.2: prozac ===
               condition  n_prescripciones  avg_rating
4             depression                36         7.5
1                anxiety                 2         9.0
0         antidepressant                 1         4.0
2  anxiety, hopelessness                 1         8.0
3              depressio                 1         7.0


In [9]:
from wordcloud import WordCloud
import re

def generar_wordcloud(condition, df, save_name):
    condition = condition.lower().strip()
    subset = df[df['condition'] == condition]
    if subset.empty:
        print(f"Sin datos de reseñas para: {condition}")
        return
    
    # Combinar las tres reseñas del dataset de DrugLib
    texto = ' '.join(subset['benefitsReview'].dropna().astype(str).tolist() + 
                     subset['sideEffectsReview'].dropna().astype(str).tolist() + 
                     subset['commentsReview'].dropna().astype(str).tolist())
    
    # Limpiar texto HTML y símbolos
    texto = re.sub(r'&#\d+;', ' ', texto)
    texto = re.sub(r'[^a-zA-Z\s]', ' ', texto)
    
    wc = WordCloud(width=800, height=400, background_color='white', max_words=80, colormap='viridis').generate(texto)
    plt.figure(figsize=(10, 5))
    plt.imáshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Palabras Clave en Reseñas — {condition.title()}')
    plt.tight_layout()
    plt.savefig(f'../outputs/wordcloud_{save_name}.png', dpi=150)
    plt.show()

generar_wordcloud('depression', df, 'depression')
generar_wordcloud('anxiety', df, 'anxiety')
